# FakeBDTeen Multimodal Deepfake Training (Google Colab)

This notebook is configured for Google Colab and runs:
- MobileViT-Small video branch
- Wav2Vec2-Base audio branch
- Late fusion MLP

In [ ]:
import os
import sys
import json
import shutil
import subprocess
import torch
from pathlib import Path

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Capability:', torch.cuda.get_device_capability(0))


def run_streamed(cmd):
    """Run a command and stream output continuously in notebook cells."""
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    if cmd and cmd[0] == 'python' and '-u' not in cmd:
        cmd = ['python', '-u'] + cmd[1:]

    print('Running:', ' '.join(map(str, cmd)))
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)

    print('\nCommand completed successfully.')

## Optional: Mount Google Drive

Enable this if your scripts or dataset are stored in Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg
!pip -q install --upgrade pip
!pip -q install transformers opencv-python-headless scikit-learn tqdm

## Configure Paths

Update these paths for your Colab environment.

In [ ]:
CONFIG = {
    # Root directory where training scripts will be copied/executed
    'work_dir': '/content',

    # Paths with and without spaces are both included to make resolution robust.
    'dataset_candidates': [
        '/content/drive/MyDrive/DEEPfake Papers/FakeBDTeen',
        '/content/drive/MyDrive/DEEPfake Papers/FakeBDTeen/FakeBDTeen',
        '/content/drive/MyDrive/DEEPfake_Papers/FakeBDTeen',
        '/content/drive/MyDrive/FakeBDTeen',
        '/content/FakeBDTeen',
    ],

    # Where your three scripts are currently located
    'script_candidates': [
        '/content',
        '/content/drive/MyDrive/DEEPfake Papers/FakeBDTeen',
        '/content/drive/MyDrive/DEEPfake Papers/FakeBDTeen/FakeBDTeen',
        '/content/drive/MyDrive/DEEPfake_Papers/FakeBDTeen',
        '/content/drive/MyDrive/FakeBDTeen',
        '/content/FakeBDTeen',
    ],
}

drive_root = Path('/content/drive/MyDrive')
if drive_root.exists():
    persist_root = drive_root / 'fakebdteen_colab_outputs'
    print('Drive detected. Persisting outputs to:', persist_root)
else:
    persist_root = Path('/content/fakebdteen_colab_outputs')
    print('Drive not detected. Persisting outputs to local runtime:', persist_root)

persist_root.mkdir(parents=True, exist_ok=True)
CONFIG['persist_root'] = str(persist_root)
CONFIG['video_output_dir'] = str(persist_root / 'outputs_video')
CONFIG['audio_output_dir'] = str(persist_root / 'outputs_audio')
CONFIG['fusion_output_dir'] = str(persist_root / 'outputs_fusion')

print(json.dumps(CONFIG, indent=2))

In [ ]:
VIDEO_SCRIPT = 'train_mobilevit_video_deepfake.py'
AUDIO_SCRIPT = 'train_wav2vec2_audio_deepfake.py'
FUSION_SCRIPT = 'train_fusion_mlp.py'

def has_mp4(root: Path) -> bool:
    return any(root.rglob('*.mp4')) or any(root.rglob('*.MP4'))

def resolve_dataset_root(candidates):
    for c in candidates:
        p = Path(c)
        if not p.exists():
            continue
        if has_mp4(p):
            return p
        # Try one level deeper in case of nested extraction/folder layout.
        for child in p.iterdir():
            if child.is_dir() and has_mp4(child):
                return child
    raise FileNotFoundError('No valid dataset root found with mp4 files. Update CONFIG[dataset_candidates].')

def find_script(script_name, candidates):
    for c in candidates:
        p = Path(c) / script_name
        if p.exists():
            return p
    for c in candidates:
        root = Path(c)
        if root.exists():
            hits = list(root.rglob(script_name))
            if hits:
                return hits[0]
    raise FileNotFoundError(f'{script_name} not found in script candidates')

def safe_copy2(src: Path, dst: Path) -> None:
    src_resolved = src.resolve()
    dst_resolved = dst.resolve() if dst.exists() else dst
    if src_resolved == dst_resolved:
        print(f'Skip copy (same file): {src}')
        return
    shutil.copy2(src, dst)

dataset_root = resolve_dataset_root(CONFIG['dataset_candidates'])
work_dir = Path(CONFIG['work_dir'])
work_dir.mkdir(parents=True, exist_ok=True)

video_src = find_script(VIDEO_SCRIPT, CONFIG['script_candidates'])
audio_src = find_script(AUDIO_SCRIPT, CONFIG['script_candidates'])
fusion_src = find_script(FUSION_SCRIPT, CONFIG['script_candidates'])

video_dst = work_dir / VIDEO_SCRIPT
audio_dst = work_dir / AUDIO_SCRIPT
fusion_dst = work_dir / FUSION_SCRIPT

safe_copy2(video_src, video_dst)
safe_copy2(audio_src, audio_dst)
safe_copy2(fusion_src, fusion_dst)

# Ensure output roots exist up front.
Path(CONFIG['video_output_dir']).mkdir(parents=True, exist_ok=True)
Path(CONFIG['audio_output_dir']).mkdir(parents=True, exist_ok=True)
Path(CONFIG['fusion_output_dir']).mkdir(parents=True, exist_ok=True)

print('Dataset root:', dataset_root)
print('Video script:', video_dst)
print('Audio script:', audio_dst)
print('Fusion script:', fusion_dst)
print('Video output dir:', CONFIG['video_output_dir'])
print('Audio output dir:', CONFIG['audio_output_dir'])
print('Fusion output dir:', CONFIG['fusion_output_dir'])

all_mp4 = list(dataset_root.rglob('*.mp4')) + list(dataset_root.rglob('*.MP4'))
print('Total videos found:', len(all_mp4))

## Train Video Model (MobileViT-Small)

In [ ]:
VIDEO_CONFIG = {
    'dataset_root': str(dataset_root),
    'output_dir': CONFIG['video_output_dir'],
    'epochs': 20,
    'batch_size': 8,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'num_workers': 2,
}
print(json.dumps(VIDEO_CONFIG, indent=2))

video_cmd = [
    'python', '/content/train_mobilevit_video_deepfake.py',
    '--dataset-root', VIDEO_CONFIG['dataset_root'],
    '--output-dir', VIDEO_CONFIG['output_dir'],
    '--epochs', str(VIDEO_CONFIG['epochs']),
    '--batch-size', str(VIDEO_CONFIG['batch_size']),
    '--lr', str(VIDEO_CONFIG['lr']),
    '--weight-decay', str(VIDEO_CONFIG['weight_decay']),
    '--num-workers', str(VIDEO_CONFIG['num_workers']),
]
run_streamed(video_cmd)

In [ ]:
video_out = Path(VIDEO_CONFIG['output_dir'])
print('Video output exists:', video_out.exists())
if video_out.exists():
    for p in sorted(video_out.glob('*')):
        print(' -', p.name)

video_metrics_path = video_out / 'metrics.pt'
if video_metrics_path.exists():
    video_metrics = torch.load(video_metrics_path, map_location='cpu')
    print('Video best val acc:', video_metrics.get('best_val_acc'))
else:
    print('Video metrics.pt not found yet')

## Train Audio Model (Wav2Vec2-Base)

In [ ]:
AUDIO_CONFIG = {
    'dataset_root': str(dataset_root),
    'output_dir': CONFIG['audio_output_dir'],
    'epochs': 20,
    'batch_size': 16,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'num_workers': 0,
    'ffmpeg_timeout': 45,
    'force_cpu': False,
}
print(json.dumps(AUDIO_CONFIG, indent=2))

audio_cmd = [
    'python', '/content/train_wav2vec2_audio_deepfake.py',
    '--dataset-root', AUDIO_CONFIG['dataset_root'],
    '--output-dir', AUDIO_CONFIG['output_dir'],
    '--epochs', str(AUDIO_CONFIG['epochs']),
    '--batch-size', str(AUDIO_CONFIG['batch_size']),
    '--lr', str(AUDIO_CONFIG['lr']),
    '--weight-decay', str(AUDIO_CONFIG['weight_decay']),
    '--num-workers', str(AUDIO_CONFIG['num_workers']),
    '--ffmpeg-timeout', str(AUDIO_CONFIG['ffmpeg_timeout']),
]
if AUDIO_CONFIG.get('force_cpu', False):
    audio_cmd.append('--force-cpu')

run_streamed(audio_cmd)

In [ ]:
audio_out = Path(AUDIO_CONFIG['output_dir'])
print('Audio output exists:', audio_out.exists())
if audio_out.exists():
    for p in sorted(audio_out.glob('*')):
        print(' -', p.name)

audio_metrics_path = audio_out / 'metrics.pt'
if audio_metrics_path.exists():
    audio_metrics = torch.load(audio_metrics_path, map_location='cpu')
    print('Audio best val acc:', audio_metrics.get('best_val_acc'))
else:
    print('Audio metrics.pt not found yet')

## Train Late Fusion MLP

In [ ]:
FUSION_CONFIG = {
    'output_dir': CONFIG['fusion_output_dir'],
    'epochs': 30,
    'batch_size': 32,
    'lr': 5e-4,
    'weight_decay': 1e-2,
}

video_out = Path(VIDEO_CONFIG['output_dir'])
audio_out = Path(AUDIO_CONFIG['output_dir'])

video_train = video_out / 'train_embeddings.pt'
video_val = video_out / 'val_embeddings.pt'
video_test = video_out / 'test_embeddings.pt'

audio_train = audio_out / 'train_embeddings.pt'
audio_val = audio_out / 'val_embeddings.pt'
audio_test = audio_out / 'test_embeddings.pt'

required = [video_train, video_val, video_test, audio_train, audio_val, audio_test]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing embeddings:\n' + '\n'.join(missing))

fusion_cmd = [
    'python', '/content/train_fusion_mlp.py',
    '--video-train-embed', str(video_train),
    '--audio-train-embed', str(audio_train),
    '--video-val-embed', str(video_val),
    '--audio-val-embed', str(audio_val),
    '--video-test-embed', str(video_test),
    '--audio-test-embed', str(audio_test),
    '--output-dir', FUSION_CONFIG['output_dir'],
    '--epochs', str(FUSION_CONFIG['epochs']),
    '--batch-size', str(FUSION_CONFIG['batch_size']),
    '--lr', str(FUSION_CONFIG['lr']),
    '--weight-decay', str(FUSION_CONFIG['weight_decay']),
]
run_streamed(fusion_cmd)

In [ ]:
fusion_out = Path(FUSION_CONFIG['output_dir'])
print('Fusion output exists:', fusion_out.exists())
if fusion_out.exists():
    for p in sorted(fusion_out.glob('*')):
        print(' -', p.name)

metrics_pt = fusion_out / 'test_metrics.pt'
metrics_json = fusion_out / 'test_metrics.json'

if metrics_pt.exists():
    m = torch.load(metrics_pt, map_location='cpu')
    print('Fusion accuracy:', m.get('accuracy'))
    print('Fusion f1:', m.get('f1'))
    print('Fusion auc_roc:', m.get('auc_roc'))
    print('Confusion matrix:', m.get('confusion_matrix'))
elif metrics_json.exists():
    print(metrics_json.read_text())
else:
    print('Fusion test metrics not found yet')

## Download Artifacts

In [ ]:
zip_path = '/content/fakebdteen_colab_outputs.zip'
video_dir = CONFIG['video_output_dir']
audio_dir = CONFIG['audio_output_dir']
fusion_dir = CONFIG['fusion_output_dir']

!zip -r {zip_path} {video_dir} {audio_dir} {fusion_dir}
print('Created:', zip_path)
# Optional:
# from google.colab import files
# files.download(zip_path)